In [1]:
import pandas as pd
import numpy as np
import requests

In [2]:
df = pd.read_csv("uber.csv")

print(df.shape)
df.head()

(200000, 9)


,Unnamed: 0,key,fare_amount,pickup_datetime,pickup_longitude,pickup_latitude,dropoff_longitude,dropoff_latitude,passenger_count
0,24238194,2015-05-07 19:52:06.0000003,7.5,2015-05-07 19:52:06 UTC,-73.999817,40.738354,-73.999512,40.723217,1
1,27835199,2009-07-17 20:04:56.0000002,7.7,2009-07-17 20:04:56 UTC,-73.994355,40.728225,-73.994710,40.750325,1
2,44984355,2009-08-24 21:45:00.00000061,12.9,2009-08-24 21:45:00 UTC,-74.005043,40.740770,-73.962565,40.772647,1
3,25894730,2009-06-26 08:22:21.0000001,5.3,2009-06-26 08:22:21 UTC,-73.976124,40.790844,-73.965316,40.803349,3
4,17610152,2014-08-28 17:47:00.000000188,16.0,2014-08-28 17:47:00 UTC,-73.925023,40.744085,-73.973082,40.761247,5


In [3]:
# remove unwanted index column if exists
df = df.loc[:, ~df.columns.str.contains('^Unnamed')]

# drop missing datetime rows
df = df.dropna(subset=['pickup_datetime'])

df.head()

,key,fare_amount,pickup_datetime,pickup_longitude,pickup_latitude,dropoff_longitude,dropoff_latitude,passenger_count
0,2015-05-07 19:52:06.0000003,7.5,2015-05-07 19:52:06 UTC,-73.999817,40.738354,-73.999512,40.723217,1
1,2009-07-17 20:04:56.0000002,7.7,2009-07-17 20:04:56 UTC,-73.994355,40.728225,-73.994710,40.750325,1
2,2009-08-24 21:45:00.00000061,12.9,2009-08-24 21:45:00 UTC,-74.005043,40.740770,-73.962565,40.772647,1
3,2009-06-26 08:22:21.0000001,5.3,2009-06-26 08:22:21 UTC,-73.976124,40.790844,-73.965316,40.803349,3
4,2014-08-28 17:47:00.000000188,16.0,2014-08-28 17:47:00 UTC,-73.925023,40.744085,-73.973082,40.761247,5


In [4]:
df['pickup_datetime'] = pd.to_datetime(
    df['pickup_datetime'],
    utc=True,
    errors='coerce'
)

# create hourly bucket
df['pickup_hour'] = df['pickup_datetime'].dt.floor('h')

# remove timezone for merge compatibility
df['pickup_hour'] = df['pickup_hour'].dt.tz_localize(None)

print(df[['pickup_datetime','pickup_hour']].head())
print(df['pickup_hour'].dtype)

            pickup_datetime         pickup_hour
0 2015-05-07 19:52:06+00:00 2015-05-07 19:00:00
1 2009-07-17 20:04:56+00:00 2009-07-17 20:00:00
2 2009-08-24 21:45:00+00:00 2009-08-24 21:00:00
3 2009-06-26 08:22:21+00:00 2009-06-26 08:00:00
4 2014-08-28 17:47:00+00:00 2014-08-28 17:00:00
datetime64[ns]


In [5]:
def get_weather_bulk(start, end, lat, lon):

    url = "https://archive-api.open-meteo.com/v1/archive"

    params = {
        "latitude": lat,
        "longitude": lon,
        "start_date": start,
        "end_date": end,
        "hourly": "temperature_2m,precipitation,windspeed_10m",
        "timezone": "UTC"
    }

    response = requests.get(url, params=params)
    data = response.json()

    weather_df = pd.DataFrame({
        "pickup_hour": pd.to_datetime(data["hourly"]["time"]),
        "temperature": data["hourly"]["temperature_2m"],
        "precipitation": data["hourly"]["precipitation"],
        "wind_speed": data["hourly"]["windspeed_10m"]
    })

    return weather_df

In [6]:
start = df['pickup_hour'].min().strftime("%Y-%m-%d")
end = df['pickup_hour'].max().strftime("%Y-%m-%d")

weather_df = get_weather_bulk(
    start,
    end,
    lat=40.7128,   # NYC
    lon=-74.0060
)

weather_df.head()

,pickup_hour,temperature,precipitation,wind_speed
0,2009-01-01 00:00:00,-6.7,0.0,32.8
1,2009-01-01 01:00:00,-7.2,0.0,34.1
2,2009-01-01 02:00:00,-7.3,0.0,33.6
3,2009-01-01 03:00:00,-7.4,0.0,33.9
4,2009-01-01 04:00:00,-7.4,0.0,31.3


In [7]:
weather_df['pickup_hour'] = pd.to_datetime(
    weather_df['pickup_hour']
)

df = df.merge(
    weather_df,
    on="pickup_hour",
    how="left"
)

print("Merge Successful ✅")

Merge Successful ✅


In [8]:
weather_cols = ['temperature','precipitation','wind_speed']

df = df.sort_values('pickup_hour')

df[weather_cols] = (
    df[weather_cols]
    .ffill()
    .bfill()
)

df[weather_cols].isna().sum()

temperature      0
precipitation    0
wind_speed       0
dtype: int64

In [9]:
print(df.info())
df.head()

<class 'pandas.core.frame.DataFrame'>
Index: 200000 entries, 100844 to 32718
Data columns (total 12 columns):
 #   Column             Non-Null Count   Dtype              
---  ------             --------------   -----              
 0   key                200000 non-null  object             
 1   fare_amount        200000 non-null  float64            
 2   pickup_datetime    200000 non-null  datetime64[ns, UTC]
 3   pickup_longitude   200000 non-null  float64            
 4   pickup_latitude    200000 non-null  float64            
 5   dropoff_longitude  199999 non-null  float64            
 6   dropoff_latitude   199999 non-null  float64            
 7   passenger_count    200000 non-null  int64              
 8   pickup_hour        200000 non-null  datetime64[ns]     
 9   temperature        200000 non-null  float64            
 10  precipitation      200000 non-null  float64            
 11  wind_speed         200000 non-null  float64            
dtypes: datetime64[ns, UTC](1), date

,key,fare_amount,pickup_datetime,pickup_longitude,pickup_latitude,dropoff_longitude,dropoff_latitude,passenger_count,pickup_hour,temperature,precipitation,wind_speed
100844,2009-01-01 01:15:22.0000006,8.5,2009-01-01 01:15:22+00:00,-73.981918,40.779456,-73.957685,40.771043,2,2009-01-01 01:00:00,-7.2,0.0,34.1
43961,2009-01-01 01:59:17.0000001,13.0,2009-01-01 01:59:17+00:00,-73.983759,40.721389,-73.994833,40.687179,2,2009-01-01 01:00:00,-7.2,0.0,34.1
7628,2009-01-01 02:05:03.0000003,10.6,2009-01-01 02:05:03+00:00,-73.956635,40.771254,-73.991528,40.749778,2,2009-01-01 02:00:00,-7.3,0.0,33.6
134679,2009-01-01 02:14:20.0000003,5.0,2009-01-01 02:14:20+00:00,-73.986486,40.734734,-73.983508,40.730138,1,2009-01-01 02:00:00,-7.3,0.0,33.6
118760,2009-01-01 02:09:13.0000003,12.2,2009-01-01 02:09:13+00:00,-73.984605,40.728020,-73.955746,40.776830,1,2009-01-01 02:00:00,-7.3,0.0,33.6


In [10]:
df.to_csv(
    "uber_weather_enriched.csv",
    index=False
)

print("Dataset exported successfully ✅")

Dataset exported successfully ✅
